In [49]:
import pandas as pd
from datetime import date, timedelta
from datetime import datetime
import os
import shutil
import pathlib
from nbconvert import PythonExporter
import polars as pl
import glob
import time

In [50]:
def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])


def input_data(data_dir):
    """Read all xlsx/csv files in a directory into a single Polars DataFrame."""
    
    def read_file(filename):
        try:
            mtime = convert_to_datetime(time.localtime(os.path.getmtime(filename)))
            df = (pl.read_excel(filename, infer_schema_length=0) if filename.suffix.lower() == '.xlsx'
                  else pl.read_csv(filename, infer_schema_length=0,
                                   encoding='utf-8' if True else 'ISO-8859-1'))
            return df.with_columns(pl.lit(filename.stem).alias('sheet_name'),
                                   pl.lit(mtime).alias('Export time'))
        except Exception:
            try:
                return (pl.read_csv(filename, infer_schema_length=0,
                                    encoding='ISO-8859-1', ignore_errors=True)
                        .with_columns(pl.lit(filename.stem).alias('sheet_name'),
                                      pl.lit(mtime).alias('Export time')))
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                return None

    dfs = [df for f in pathlib.Path(data_dir).glob('**/*.*')
           if f.suffix.lower() in ('.xlsx', '.csv') and (df := read_file(f)) is not None]
    
    if not dfs:
        return pl.DataFrame()

    col_dtypes = {col: {df[col].dtype for df in dfs if col in df.columns}
                  for col in set(col for df in dfs for col in df.columns)}
    dfs = [df.with_columns(pl.col(c).cast(pl.Utf8)
                           for c, t in col_dtypes.items() if c in df.columns and len(t) > 1)
           for df in dfs]

    return pl.concat(dfs, how='vertical')

In [51]:
notebook_dict = {
  'IEX_optimized.ipynb': 1,
  'RTA_optimized.ipynb': 1,
}

def convert_notebook(notebook_path):
    exporter = PythonExporter()
    output, _ = exporter.from_filename(notebook_path)
    return output

first_glob = os.path.expanduser("~").replace("\\", "/")
test_path  = f'{first_glob}/Concentrix Corporation'
if not os.path.exists(test_path):
    raise FileNotFoundError(f'Not found the path: {test_path}')

folder_paths = {
    'output_iex_base'     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE'
}

print(f"Currently using base directory: {first_glob}")
IEX_BASE = input_data(folder_paths['output_iex_base'])
IEX_BASE


Currently using base directory: C:/Users/ADMIN


Agent,Date,Start_Shift,End_Shift,Scheduled Activity,Start_Action,End_Action,sheet_name,Export time,Generate Date
str,str,str,str,str,str,str,str,datetime[μs],str
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Break""","""10:15 PM""","""10:30 PM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Open Time""","""10:30 PM""","""2:30 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Lunch""","""2:30 AM""","""3:30 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Open Time""","""3:30 AM""","""4:30 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Break""","""4:30 AM""","""4:45 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Open Time""","""4:45 AM""","""6:00 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Extra Hours""","""6:00 AM""","""7:45 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Break""","""7:45 AM""","""8:00 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""
"""Agent: 3000182 HOANG, THIPHUON…","""2026-01-05""","""9:00 PM""","""10:00 AM""","""Extra Hours""","""8:00 AM""","""10:00 AM""","""IEX_Base_2026_01_05""",2026-05-18 00:37:43,"""1/13/26 3:36 AM"""


In [52]:
for nbook in notebook_dict:
    if notebook_dict[nbook] == 0:
        print(f"Not executing notebook {nbook}.")
        continue
        
    nbook_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Python_Code/{nbook}'
    
    if os.path.isfile(nbook_path):
        print(f"Running notebook: {nbook}")
        try:
            script = convert_notebook(nbook_path)
            exec(script)
        except Exception as e:
            print(f"Error running notebook: {nbook}")
            print(str(e))
    else:
        print(f"Error: Notebook {nbook} not found.")

Running notebook: IEX_optimized.ipynb
--- FULL FOLDER PATHS LIST ---
input_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
input_hc_master: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx
output_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT
output_iex_all: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
output_iex_intervals: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS
output_iex_base: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE
hc_extend_by_month: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_iex_schedule_adjustment: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia

<string>:273: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:307: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


✅ Exported 20 file(s) to 'C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE':
   IEX_Base_2026_01_05.csv  (4050 rows)
   IEX_Base_2026_01_12.csv  (4063 rows)
   IEX_Base_2026_01_19.csv  (3906 rows)
   IEX_Base_2026_01_26.csv  (3727 rows)
   IEX_Base_2026_02_02.csv  (3738 rows)
   IEX_Base_2026_02_09.csv  (3619 rows)
   IEX_Base_2026_02_16.csv  (3682 rows)
   IEX_Base_2026_02_23.csv  (3032 rows)
   IEX_Base_2026_03_02.csv  (3208 rows)
   IEX_Base_2026_03_09.csv  (2498 rows)
   IEX_Base_2026_03_16.csv  (3008 rows)
   IEX_Base_2026_03_23.csv  (2994 rows)
   IEX_Base_2026_03_30.csv  (2669 rows)
   IEX_Base_2026_04_06.csv  (2573 rows)
   IEX_Base_2026_04_13.csv  (3197 rows)
   IEX_Base_2026_04_20.csv  (3372 rows)
   IEX_Base_2026_04_27.csv  (3151 rows)
   IEX_Base_2026_05_04.csv  (3800 rows)
   IEX_Base_2026_05_11.csv  (3493 rows)
   IEX_Base_2026_05_18.csv  (3981 rows)
Valid weeks found: ['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-

<string>:358: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


Rows after filter: 67761
Weeks retained: ['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02', '2026-02-09', '2026-02-16', '2026-02-23', '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23', '2026-03-30', '2026-04-06', '2026-04-13', '2026-04-20', '2026-04-27', '2026-05-04', '2026-05-11', '2026-05-18']


<string>:742: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


Valid weeks: [Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-19 00:00:00'), Timestamp('2026-01-26 00:00:00'), Timestamp('2026-02-02 00:00:00'), Timestamp('2026-02-09 00:00:00'), Timestamp('2026-02-16 00:00:00'), Timestamp('2026-02-23 00:00:00'), Timestamp('2026-03-02 00:00:00'), Timestamp('2026-03-09 00:00:00'), Timestamp('2026-03-16 00:00:00'), Timestamp('2026-03-23 00:00:00'), Timestamp('2026-03-30 00:00:00'), Timestamp('2026-04-06 00:00:00'), Timestamp('2026-04-13 00:00:00'), Timestamp('2026-04-20 00:00:00'), Timestamp('2026-04-27 00:00:00'), Timestamp('2026-05-04 00:00:00'), Timestamp('2026-05-11 00:00:00'), Timestamp('2026-05-18 00:00:00')]
Running notebook: RTA_optimized.ipynb
--- FULL FOLDER PATHS LIST ---
input_iex_rawdata: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
storage_iex_rawdata: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPU

Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 8, falling back to string
<string>:388: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`


shape: (1_882, 7)
┌───────────┬────────────┬─────────┬───────┬───────────────┬──────────────┬────────────────────────┐
│ EID       ┆ Date       ┆ ot_type ┆ hours ┆ nsa_authorize ┆ ot_authorize ┆ ot_status              │
│ ---       ┆ ---        ┆ ---     ┆ ---   ┆ ---           ┆ ---          ┆ ---                    │
│ str       ┆ date       ┆ str     ┆ f64   ┆ str           ┆ str          ┆ str                    │
╞═══════════╪════════════╪═════════╪═══════╪═══════════════╪══════════════╪════════════════════════╡
│ 102871040 ┆ 2026-04-30 ┆ OT3.0X  ┆ 8.0   ┆               ┆ Authorized   ┆ (8.0 hrs - Authorized) │
│ 102457937 ┆ 2025-09-02 ┆ OT3.9X  ┆ 8.0   ┆               ┆ Authorized   ┆ (8.0 hrs - Authorized) │
│ 102871040 ┆ 2026-02-05 ┆ OT2.0X  ┆ 3.5   ┆ Authorized    ┆ Authorized   ┆ (3.5 hrs - Authorized) │
│ 103173195 ┆ 2026-05-05 ┆ OT1.5X  ┆ 2.0   ┆ Authorized    ┆ Authorized   ┆ (2.0 hrs - Authorized) │
│ 102477510 ┆ 2026-02-18 ┆ OT3.0X  ┆ 8.0   ┆               ┆ Authorized  

In [53]:
def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, infer_schema_length=0)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)
        
        df = df.with_columns([
            pl.col(col).cast(pl.String) 
            for col in df.columns
        ])
        
        df_list.append(df)
    
    merged_df = pl.concat(df_list, how='vertical')

    return merged_df

In [54]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(100)

def filter_shift_discrepancies(df_input) -> pl.DataFrame:
    if isinstance(df_input, pd.DataFrame):
        df = pl.from_pandas(df_input)
    else:
        df = df_input

    # Requirement 1: hc_actual != hc_present
    hc_mismatch_condition = (
        pl.col("hc_actual").cast(pl.Float64, strict=False).fill_null(0.0) != 
        pl.col("hc_present").cast(pl.Float64, strict=False).fill_null(0.0)
    )

    # Requirement 2: Target > 3.75 AND sum_productive <= 3
    productivity_condition = (
        (pl.col("Target").cast(pl.Float64, strict=False).fill_null(0.0) > 3.75) &
        (pl.col("sum_productive").cast(pl.Float64, strict=False).fill_null(0.0) <= 3.0)
    )

    # Exclude any LOB containing "Support" (e.g. "Support", "Support - Ops", etc.)
    lob_condition = ~pl.col("LOB").str.contains("Support", literal=True)

    # Exclude Training/Offline shifts, Off Phone Misc, and PO from Shift Tracking
    shift_condition = (
        (~pl.col("First Shift").is_in(["Training", "Offline"])) &
        (~pl.col("Shift Tracking").is_in(["Training Offline", "Off Phone Misc", "PO"]))
    )

    # Main filtering logic
    result = df.filter(
        (hc_mismatch_condition | productivity_condition) &
        lob_condition &
        shift_condition
    )

    if "Date_Converted" in result.columns:
        result = result.sort("Date_Converted")

    return result

RTA_REPORT_GENERATE = input_data(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA')
print(RTA_REPORT_GENERATE.columns)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Date_Converted').str.slice(0, 10).str.strptime(pl.Date, format='%Y-%m-%d'),
    pl.col('Target').cast(pl.Float64), 
    pl.col('duration').cast(pl.Float64),
    pl.col('sum_productive').cast(pl.Float64),
    pl.col('start').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col('end').str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False)
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.filter(
    pl.col('duration').is_not_null() | (pl.col('Target') > 0)
)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col('Shift Tracking').cast(pl.Utf8).str.strip_chars(),
    pl.col('Open Time').cast(pl.Float64).fill_null(0.0), 
    pl.col('Extra Time').cast(pl.Float64).fill_null(0.0)    
])
 
RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.when(pl.col('Shift Tracking') == 'HDL').then(0.5)
    .when(
        (pl.col('Shift Tracking').is_in(['PR','PR - OT','PO'])) & 
        ((pl.col('Open Time') > 0) | (pl.col('Extra Time') > 0))
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking').is_in([
            'Training Offline', 'Sick Leave', 'Paid Leave', 'Billable Training'
        ])) & (pl.col('Open Time') == 0)
    ).then(1.0)
    .when(
        (pl.col('Shift Tracking') != 'HDL') & 
        (pl.col('Shift Tracking') != 'PR') & 
        (pl.col('Open Time') == 0)
    ).then(0.0)
    .otherwise(None)
    .alias('hc_present')
])
 
filtered_data = RTA_REPORT_GENERATE.select([
    'Date_Converted', 'Employee Name', 'LOB', 'Detail Status','IEX ID','First Shift','Shift Tracking', 
    'start', 'end','Target', 'duration', 'sum_productive','training-idle','Extra Time', 'hc_schedule',
    'hc_actual', 'hc_present'
])
filtered_data = filtered_data.with_columns([
    pl.col('hc_actual').cast(pl.Float64),
    pl.col('hc_present').cast(pl.Float64)
])
 
start_date = '2026-03-01'
end_date = '2026-03-31'
 
start_date_parsed = pl.lit(start_date).str.strptime(pl.Date, format='%Y-%m-%d')
end_date_parsed = pl.lit(end_date).str.strptime(pl.Date, format='%Y-%m-%d')
 
sorted_data = filtered_data.sort('Date_Converted')
filtered_data = sorted_data.filter((pl.col('Date_Converted') >= start_date_parsed) & (pl.col('Date_Converted') <= end_date_parsed))
 
filtered_data_pd = filtered_data.to_pandas()

filtered_data_1 = filter_shift_discrepancies(filtered_data)
filtered_data_1

['Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id', 'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled', 'Night_Shift', 'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift', 'start', 'end', 'total_time_chat_handle', 'sum_productive', 'break', 'lunch', 'coaching-idle', 'training-idle', 'outbound-idle', 'break_count', 'other_status', 'over_break', 'over_lunch', 'exceed_break', 'duration', 'hc_actual', 'hc_schedule', 'time_late', 'time_leave', 'lateness', 'adherence_time', 'total_time_pull_out', 'ramco_marked', 'nsa_authorize', 'ot_authorize', 'hours', 'ot_

Date_Converted,Employee Name,LOB,Detail Status,IEX ID,First Shift,Shift Tracking,start,end,Target,duration,sum_productive,training-idle,Extra Time,hc_schedule,hc_actual,hc_present
date,str,str,str,str,str,str,datetime[ms],datetime[ms],f64,f64,f64,str,f64,str,f64,f64
2026-03-03,"""NGUYEN NGOC SON""","""Lodging""","""Lodging Production""","""3089107""","""0700-1600""","""NCNS""",2026-03-03 06:57:09,2026-03-03 16:12:09,0.0,9.249963,0.588991,"""0.0""",0.0,"""1.0""",1.0,0.0
2026-03-08,"""PHAN DAT LOI""","""Lodging""","""Lodging Production""","""3092114""","""2200-0700""","""HDL""",2026-03-08 22:20:55,2026-03-09 14:56:28,3.75,16.592724,11.035701,"""0.0006905555725097656""",0.0,"""1.0""",1.0,0.5
2026-03-08,"""BUI BA DUONG""","""Lodging""","""Lodging Production""","""3052306""","""2100-0600""","""AL""",2026-03-09 04:51:09,2026-03-09 14:37:44,0.0,9.776337,8.130975,"""0.0""",0.0,"""1.0""",1.0,0.0
2026-03-12,"""NGUYEN MINH NHAT""","""Lodging""","""Lodging Production""","""3097284""","""0700-0430""","""Extra Hours""",2026-03-12 21:55:41,2026-03-13 09:01:11,0.0,11.091497,9.130765,"""0.0""",4.25,"""1.0""",1.0,0.0
2026-03-17,"""LE HOANG THAO NGOC""","""Non_Lodging""","""Non Lodging Production""","""3085321""","""0600-1500""","""NCNS""",2026-03-17 06:08:09,2026-03-17 09:23:12,0.0,3.250652,2.473587,"""0.43605055067274306""",0.0,"""1.0""",0.5,0.0
2026-03-20,"""VI LE HOAI ANH""","""Lodging""","""Lodging Production""","""3089071""","""2200-0700""","""PR""",2026-03-20 22:25:19,2026-03-21 07:19:33,7.5,8.904129,0.0,"""7.433989935980903""",0.0,"""1.0""",1.0,1.0
2026-03-22,"""NGUYEN THI THU THUY""","""Lodging""","""Lodging Production""","""3112804""","""0500-1400""","""NCNS""",2026-03-22 09:57:48,2026-03-22 14:08:47,0.0,4.183096,3.917133,"""0.0""",0.0,"""1.0""",0.5,0.0
2026-03-23,"""DUONG CONG HOANG""","""Lodging""","""Lodging Production""","""3089163""","""0600-0500""","""Extra Hours""",2026-03-23 20:54:06,2026-03-24 06:15:07,0.0,9.350303,9.018289,"""0.0""",3.0,"""1.0""",1.0,0.0
2026-03-23,"""TRAN THI TUONG VY""","""Lodging""","""Lodging Production""","""3087859""","""0600-1500""","""PR""",2026-03-23 05:51:42,2026-03-23 09:54:32,7.5,4.047308,3.80214,"""0.0""",0.0,"""1.0""",0.5,1.0


In [55]:
# ── FILTER IEX_BASE BY filtered_data_1 ───────────────────────────────────────

IEX_BASE_filtered = IEX_BASE.with_columns(
    pl.col("Agent").str.extract(r"Agent:\s*(\d+)", 1).alias("IEX_ID"),
    pl.col("Date").cast(pl.Utf8)
)
filter_keys = (
    filtered_data_1
    .select([
        pl.col("IEX ID").cast(pl.Utf8).alias("IEX_ID"),
        pl.col("Date_Converted").cast(pl.Utf8).alias("Date")
    ])
    .unique()
)
IEX_BASE_filtered = IEX_BASE_filtered.join(filter_keys, on=["IEX_ID", "Date"], how="semi")
IEX_BASE_filtered = IEX_BASE_filtered.with_columns(
    pl.col("Date").str.strptime(pl.Date, format="%Y-%m-%d", strict=False),
    pl.col("Generate Date").str.strptime(pl.Datetime, format="%m/%d/%y %I:%M %p", strict=False),
    pl.col("Start_Shift").str.strptime(pl.Time, format="%I:%M %p", strict=False),
    pl.col("End_Shift").str.strptime(pl.Time, format="%I:%M %p", strict=False),
    pl.lit(None).cast(pl.Utf8).alias("Scheduled Activity"),
    pl.lit(None).cast(pl.Utf8).alias("Start_Action"),
    pl.lit(None).cast(pl.Utf8).alias("End_Action"),
    pl.col("sheet_name").str.strip_chars('"').str.replace(r"IEX_Base_", "").alias("sheet_name")
).drop("IEX_ID").unique().sort("Date")

with pl.Config(tbl_rows=50, tbl_cols=20, fmt_str_lengths=100, tbl_width_chars=500):
    display(IEX_BASE_filtered)

Agent,Date,Start_Shift,End_Shift,Scheduled Activity,Start_Action,End_Action,sheet_name,Export time,Generate Date
str,date,time,time,str,str,str,str,datetime[μs],datetime[μs]
"""Agent: 3089107 Nguyen, NgocSon""",2026-03-03,07:00:00,16:00:00,null,null,null,"""2026_03_02""",2026-05-18 00:37:43,2026-03-10 02:52:00
"""Agent: 3092114 Phan, DatLoi""",2026-03-08,22:00:00,07:00:00,null,null,null,"""2026_03_02""",2026-05-18 00:37:43,2026-03-10 02:52:00
"""Agent: 3052306 BUI, BADUONG""",2026-03-08,21:00:00,06:00:00,null,null,null,"""2026_03_02""",2026-05-18 00:37:43,null
"""Agent: 3097284 Nguyen, MinhNhat""",2026-03-12,07:00:00,07:00:00,null,null,null,"""2026_03_09""",2026-05-18 00:37:43,2026-03-17 03:22:00
"""Agent: 3085321 LE, HOANGTHAONGOC""",2026-03-17,06:00:00,15:00:00,null,null,null,"""2026_03_16""",2026-05-18 00:37:43,2026-03-22 03:25:00
"""Agent: 3089071 Vi, LeHoaiAnh""",2026-03-20,22:00:00,07:00:00,null,null,null,"""2026_03_16""",2026-05-18 00:37:43,2026-03-22 03:25:00
"""Agent: 3112804 Nguyen, ThiThuThuy""",2026-03-22,05:00:00,14:00:00,null,null,null,"""2026_03_16""",2026-05-18 00:37:43,null
"""Agent: 3087859 TRAN, THITUONGVY""",2026-03-23,06:00:00,15:00:00,null,null,null,"""2026_03_23""",2026-05-18 00:37:43,2026-04-07 03:41:00
"""Agent: 3089163 Duong, CongHoang""",2026-03-23,06:00:00,06:00:00,null,null,null,"""2026_03_23""",2026-05-18 00:37:43,2026-04-07 03:41:00


In [56]:
# def get_xlsx_filenames(data_dir):
#     xlsx_files = []
#     for filename in pathlib.Path(data_dir).glob('**/*.xlsx'):
#         xlsx_files.append(filename.name)
#     return xlsx_files
# def create_dataframe_with_filenames(data_dir):
#     xlsx_filenames = get_xlsx_filenames(data_dir)
#     df = pd.DataFrame(xlsx_filenames, columns=['Filename'])
#     return df
# def get_latest_xlsx_files(df):
#     df = df[df['Filename'].str.match(r'\d{4}-\d{2}-\d{2}\.xlsx')]
#     df['Date'] = pd.to_datetime(df['Filename'].str.replace('.xlsx', ''), format='%Y-%m-%d')
#     df_sorted = df.sort_values(by='Date', ascending=False)
#     latest_files = df_sorted.head(8)['Filename'].tolist()
#     return latest_files
# def delete_all_xlsx_files(directory):
#     for filename in pathlib.Path(directory).glob('*.xlsx'):
#         os.remove(filename)
# def copy_latest_files(source_dir, dest_dir, latest_files):
#     for filename in latest_files:
#         source_path = pathlib.Path(source_dir) / filename
#         dest_path = pathlib.Path(dest_dir) / filename
#         shutil.copy(source_path, dest_path)

# source_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA'
# dest_dir = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_RTA_FOR_REPORT'

# df = create_dataframe_with_filenames(source_dir)
# latest_files = get_latest_xlsx_files(df)
# delete_all_xlsx_files(dest_dir)
# copy_latest_files(source_dir, dest_dir, latest_files)